<a href="https://colab.research.google.com/github/fioremartiro/Machine-Learning/blob/main/Supervised%20Learning/Classification%20Examples/2.%20linear_discriminant_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 0. Import dependencies

In [27]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

### 1. Load the dataset

In [28]:
breast_cancer = load_breast_cancer()
X = breast_cancer.data # Features
y = breast_cancer.target # Output

### 2. Split into Train and Test sets properly

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

### 3. Scale the data so the floor dimensions stay small (e.g., -3 to 3 instead of -1000 to 1500)

In [30]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 4. Reduce features to 2D using PCA first

In [31]:
# (LDA can only reduce to 1 dimension for a 2-class problem, so we use PCA to get 2 variables)
pca = PCA(n_components=2)
X_train_2d = pca.fit_transform(X_train_scaled)
X_test_2d = pca.transform(X_test_scaled)

### 5. Train the Linear Discriminant Analysis (LDA) classifier

In [32]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_2d, y_train)

LinearDiscriminantAnalysis()

### 7. Interactive 3D Plot with Plotly

To make the 3D visualization interactive and movable, we can use the `Plotly` library. This will allow you to rotate, zoom, and pan the plot directly within the notebook output.

In [33]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create the Plotly figure
fig = go.Figure()

# Add the first Gaussian hill (Malignant)
fig.add_trace(go.Surface(
    x=xx, y=yy, z=z0, colorscale='Reds', opacity=0.6,
    name='Malignant (PDF)', showscale=False,
    customdata=np.stack((xx, yy, z0), axis=-1), # Store data for hover
    hovertemplate='<b>Malignant PDF</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>Density: %{z:.2e}<extra></extra>'
))

# Add the second Gaussian hill (Benign)
fig.add_trace(go.Surface(
    x=xx, y=yy, z=z1, colorscale='YlGnBu', opacity=0.6,
    name='Benign (PDF)', showscale=False,
    customdata=np.stack((xx, yy, z1), axis=-1),
    hovertemplate='<b>Benign PDF</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>Density: %{z:.2e}<extra></extra>'
))

# Add actual test data points for Malignant (y_test == 0)
fig.add_trace(go.Scatter3d(
    x=X_test_2d[y_test == 0, 0],
    y=X_test_2d[y_test == 0, 1],
    z=np.zeros_like(X_test_2d[y_test == 0, 0]), # Plot on the 'floor'
    mode='markers', marker=dict(color='red', size=5, line=dict(color='black', width=1)),
    name='Malignant (Actual 0)',
    hovertemplate='<b>Actual Malignant</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
))

# Add actual test data points for Benign (y_test == 1)
fig.add_trace(go.Scatter3d(
    x=X_test_2d[y_test == 1, 0],
    y=X_test_2d[y_test == 1, 1],
    z=np.zeros_like(X_test_2d[y_test == 1, 0]), # Plot on the 'floor'
    mode='markers', marker=dict(color='green', size=5, line=dict(color='black', width=1)),
    name='Benign (Actual 1)',
    hovertemplate='<b>Actual Benign</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
))

# Add markers for the class means
fig.add_trace(go.Scatter3d(
    x=[mean_0[0]], y=[mean_0[1]], z=[max_z * 1.05],
    mode='markers', marker=dict(symbol='x', size=10, color='darkred', line=dict(color='white', width=1)),
    name='Malignant Mean',
    hovertemplate='<b>Malignant Mean</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
))

fig.add_trace(go.Scatter3d(
    x=[mean_1[0]], y=[mean_1[1]], z=[max_z * 1.05],
    mode='markers', marker=dict(symbol='x', size=10, color='darkgreen', line=dict(color='white', width=1)),
    name='Benign Mean',
    hovertemplate='<b>Benign Mean</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
))

# --- Decision Boundary Construction ---
boundary_line_x, boundary_line_y, boundary_line_z = [], [], []
ny, nx = grid_preds.shape

# Find horizontal boundary segments (where class changes vertically across grid cells)
for i in range(ny - 1):
    for j in range(nx):
        if grid_preds[i, j] != grid_preds[i + 1, j]:
            # A horizontal boundary segment exists between (xx[i,j], yy[i,j]) and (xx[i+1,j], yy[i+1,j])
            boundary_line_x.extend([xx[i, j], xx[i + 1, j], None])
            boundary_line_y.extend([yy[i, j], yy[i + 1, j], None])
            boundary_line_z.extend([0, 0, None])

# Find vertical boundary segments (where class changes horizontally across grid cells)
for i in range(ny):
    for j in range(nx - 1):
        if grid_preds[i, j] != grid_preds[i, j + 1]:
            # A vertical boundary segment exists between (xx[i,j], yy[i,j]) and (xx[i,j+1], yy[i,j+1])
            boundary_line_x.extend([xx[i, j], xx[i, j + 1], None])
            boundary_line_y.extend([yy[i, j], yy[i, j + 1], None])
            boundary_line_z.extend([0, 0, None])

# Add the decision boundary trace if segments were found
if boundary_line_x:
    fig.add_trace(go.Scatter3d(
        x=boundary_line_x, y=boundary_line_y, z=boundary_line_z,
        mode='lines', line=dict(color='black', width=3, dash='dash'),
        name='Decision Boundary',
        hovertemplate='<b>Decision Boundary</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
    ))

# Update layout for better aesthetics
fig.update_layout(
    title='Interactive LDA 3D Probability Distributions & Decision Boundary',
    scene=dict(
        xaxis_title='Principal Component 1',
        yaxis_title='Principal Component 2',
        zaxis_title='Probability Density',
        zaxis_range=[0, max_z * 1.15], # Set Z-axis range similar to matplotlib
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.8)) # Adjust camera angle for better view
    ),
    height=700,
    width=900,
    showlegend=True
)

fig.show()
